# arms_embed_anchor — similarity of the real target to its own references

Similarity of the real target image, the ground-truth output of each row, to the row's own references, beside the same measure for the generated sets of `arms_embed`, `positive_control_check` and `positive_control_permute`.

Per row, for an image that is either the real target or a generated output: DINOv2 and CLIP cosine to each reference, averaged over the row's references; ArcFace with each reference's first stored face best-matched against every face in the image, averaged over the references that carry a face, undefined where the image or every reference carries none. One rule for every column. Section 3 lists the rows where this differs from the largest-face values of `arms_embed` and `positive_control_check`.

Strata by `ref_kind`, from the `_up` flag in the reference filenames, as in the scorers. Gaps against the real target are paired per row, with 95% intervals from target-clustered standard errors.

Reads `Data/embeddings/*.npz`, `Data/test.jsonl`, `Data/screen.jsonl` and, for the parity check, the per-row files in `Data/scores/`.

## 1. Inputs

In [1]:
import json, math, os
from pathlib import Path
import numpy as np

# notebook folder first; the absolute path is the local copy of the Drive root
CANDIDATES = [Path.cwd(),
              Path(r"D:\Documents\My Documents\University\MSc in Artificial Intelligence and Data Engineering"
                   r"\Master's Thesis\My Master's Thesis\Master_Thesis_FULL\My Master's Thesis")]
ROOT = next((p for p in CANDIDATES if (p / "Data" / "embeddings").is_dir()), None)
assert ROOT is not None, "Data/embeddings not found under the notebook folder or the local thesis root"
DATA, EMB, SCORES = ROOT / "Data", ROOT / "Data" / "embeddings", ROOT / "Data" / "scores"
print("root:", ROOT)


def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def load_store(key):
    z = np.load(EMB / f"{key}.npz")
    return dict(n=int(z["n"][0]), dino=z["dino"].astype(np.float64), clip=z["clip"].astype(np.float64),
                face=z["face"].astype(np.float64), face_off=z["face_off"])


SETS = {
    "test":   ["baseline", "lora_r8_step1000", "lora_r8_step4000", "qlora_r8_step2000", "sp_iii_p139_step2000"],
    "screen": ["baseline", "baseline__rerun", "step100", "step500"],
}
SHORT = {"target": "real target", "baseline": "baseline", "lora_r8_step1000": "LoRA@1000",
         "lora_r8_step4000": "LoRA@4000", "qlora_r8_step2000": "QLoRA@2000", "sp_iii_p139_step2000": "SP(iii)@2000",
         "baseline__rerun": "rerun", "step100": "control@100", "step500": "control@500"}

SPLIT = {}
for split, labs in SETS.items():
    rows = read_jsonl(DATA / f"{split}.jsonl")
    tgts = sorted({r["output_image"] for r in rows})              # store order: sorted unique targets
    refs = sorted({p for r in rows for p in r["input_images"]})    # store order: sorted unique references
    tix, rix = {t: k for k, t in enumerate(tgts)}, {p: k for k, p in enumerate(refs)}
    st = {"targets": load_store(f"{split}_targets"), "refs": load_store(f"{split}_refs")}
    for lab in labs:
        st[lab] = load_store(f"{split}_gen__{lab}")
    assert st["targets"]["n"] == len(tgts) and st["refs"]["n"] == len(refs), f"{split}: store size mismatch"
    assert all(st[lab]["n"] == len(rows) for lab in labs), f"{split}: generated set size mismatch"
    SPLIT[split] = dict(rows=rows, labs=labs, st=st,
                        row_tgt=np.array([tix[r["output_image"]] for r in rows]),
                        row_ref=[[rix[p] for p in r["input_images"]] for r in rows])
    print(f"{split:6s} {len(rows)} rows | {len(tgts)} targets | {len(refs)} references | {len(labs)} generated sets")


root: d:\Documents\My Documents\University\MSc in Artificial Intelligence and Data Engineering\Master's Thesis\My Master's Thesis\Master_Thesis_FULL\My Master's Thesis
test   488 rows | 100 targets | 358 references | 5 generated sets
screen 302 rows | 62 targets | 223 references | 4 generated sets


## 2. Similarity to own references, per row

In [2]:
def ref_kind(r):
    ups = ["_up" in os.path.basename(p) for p in r["input_images"]]
    return "all_up" if all(ups) else ("no_up" if not any(ups) else "mixed")


def faces(store, k):
    return store["face"][store["face_off"][k]:store["face_off"][k + 1]]


def own_ref_sims(S, image_store, image_index):
    """Per-row DINOv2, CLIP and ArcFace similarity of an image to the row's own references.
    image_index[i] is the position of row i's image in image_store."""
    R = S["st"]["refs"]
    n = len(S["rows"])
    d, c, a = np.empty(n), np.empty(n), np.full(n, np.nan)
    for i, ks in enumerate(S["row_ref"]):
        j = image_index[i]
        d[i] = np.mean([image_store["dino"][j] @ R["dino"][k] for k in ks])
        c[i] = np.mean([image_store["clip"][j] @ R["clip"][k] for k in ks])
        img_faces = faces(image_store, j)
        if len(img_faces):
            s = [float(np.max(img_faces @ faces(R, k)[0])) for k in ks if len(faces(R, k))]
            if s:
                a[i] = np.mean(s)
    return {"dino": d, "clip": c, "arc": a}


SIM = {}
for split, S in SPLIT.items():
    S["kind"] = np.array([ref_kind(r) for r in S["rows"]])
    SIM[split] = {"target": own_ref_sims(S, S["st"]["targets"], S["row_tgt"])}
    for lab in S["labs"]:
        SIM[split][lab] = own_ref_sims(S, S["st"][lab], np.arange(len(S["rows"])))
    kinds = {k: int((S["kind"] == k).sum()) for k in ("no_up", "mixed", "all_up")}
    print(f"{split:6s} real target + {len(S['labs'])} generated sets | rows by ref_kind {kinds}")


test   real target + 5 generated sets | rows by ref_kind {'no_up': 157, 'mixed': 133, 'all_up': 198}
screen real target + 4 generated sets | rows by ref_kind {'no_up': 110, 'mixed': 73, 'all_up': 119}


## 3. Parity with the per-row files on disk

Stored-embedding values for the generated sets against the per-row files of `arms_embed` (test) and `positive_control_check` (screen), which take the largest face: maximum DINOv2 and CLIP difference, ArcFace coverage difference, ArcFace rows differing by more than 1e-3, and how many of those have a reference carrying more than one face.

In [3]:
PARITY = {
    "test":   lambda lab: (SCORES / f"test__{lab}_identity.jsonl",     "dino",     "clip",     "arcface_broad"),
    "screen": lambda lab: (SCORES / f"screen__{lab}_similarity.jsonl", "dino_ref", "clip_ref", "arc_ref"),
}
print(f"{'split':6s} {'set':13s} {'max|dDINO|':>11s} {'max|dCLIP|':>11s} {'ArcFace rows':>13s} "
      f"{'coverage diff':>14s} {'differ >1e-3':>13s} {'with multi-face ref':>20s}")
for split, S in SPLIT.items():
    R = S["st"]["refs"]
    multi = np.array([any(len(faces(R, k)) > 1 for k in ks) for ks in S["row_ref"]])
    for lab in S["labs"]:
        path, kd, kc, ka = PARITY[split](lab)
        rec = {r["i"]: r for r in read_jsonl(path)}
        assert sorted(rec) == list(range(len(S["rows"]))), f"{path.name}: row indices incomplete"
        mine = SIM[split][lab]
        dd = max(abs(mine["dino"][i] - rec[i][kd]) for i in rec)
        dc = max(abs(mine["clip"][i] - rec[i][kc]) for i in rec)
        cover = sum((rec[i][ka] is None) != bool(np.isnan(mine["arc"][i])) for i in rec)
        both = [i for i in rec if rec[i][ka] is not None and not np.isnan(mine["arc"][i])]
        diff = [i for i in both if abs(mine["arc"][i] - rec[i][ka]) > 1e-3]
        print(f"{split:6s} {SHORT[lab]:13s} {dd:>11.1e} {dc:>11.1e} {len(both):>13d} {cover:>14d} "
              f"{len(diff):>13d} {int(multi[diff].sum()):>20d}")


split  set            max|dDINO|  max|dCLIP|  ArcFace rows  coverage diff  differ >1e-3  with multi-face ref
test   baseline          7.1e-08     1.3e-07           376              0            19                   19
test   LoRA@1000         7.6e-08     1.3e-07           376              0            18                   18
test   LoRA@4000         9.3e-08     1.2e-07           376              0            18                   18
test   QLoRA@2000        9.4e-08     1.1e-07           376              0            19                   19
test   SP(iii)@2000      8.1e-08     1.2e-07           376              0            19                   19
screen baseline          6.9e-08     1.2e-07           206              0             5                    5
screen rerun             7.2e-08     1.1e-07           206              0             5                    5
screen control@100       7.4e-08     1.1e-07           206              0             5                    5
screen control@500 

## 4. Real target against each generated set, by ref_kind

Levels are mean similarity to own references. ArcFace is averaged over the rows where the real target and every generated set carry a value, so its columns share one row set per stratum. Gaps are generated set minus real target, paired per row, with a 95% interval from target-clustered (CR1) standard errors.

In [4]:
KINDS = ("all", "no_up", "mixed", "all_up")
MEASURES = (("dino", "DINOv2"), ("clip", "CLIP"), ("arc", "ArcFace"))


def clustered(d, g):
    """Mean of d and its 95% interval from CR1 cluster-robust standard errors over clusters g."""
    m = float(d.mean())
    groups = np.unique(g)
    G, n = len(groups), len(d)
    s = np.array([np.sum(d[g == k] - m) for k in groups])
    se = math.sqrt(G / (G - 1) * float(np.sum(s ** 2))) / n
    return m, m - 1.96 * se, m + 1.96 * se


for split, S in SPLIT.items():
    sets = ["target"] + S["labs"]
    arc_rows = np.all([~np.isnan(SIM[split][s]["arc"]) for s in sets], axis=0)
    for key, name in MEASURES:
        base_mask = arc_rows if key == "arc" else np.ones(len(S["rows"]), bool)
        masks = {k: base_mask & ((S["kind"] == k) if k != "all" else True) for k in KINDS}
        print(f"\n=== {split}, {name}: mean similarity to own references")
        print(f"{'set':13s}" + "".join(f"{k:>10s}" for k in KINDS))
        for s in sets:
            print(f"{SHORT[s]:13s}" + "".join(f"{SIM[split][s][key][masks[k]].mean():>10.4f}" for k in KINDS))
        print(f"{'rows':13s}" + "".join(f"{int(masks[k].sum()):>10d}" for k in KINDS))
        print(f"{'targets':13s}" + "".join(f"{len(np.unique(S['row_tgt'][masks[k]])):>10d}" for k in KINDS))
        print(f"--- {split}, {name}: generated set minus real target, mean [95% CI, target-clustered]")
        print(f"{'set':13s}" + "".join(f"{k:>27s}" for k in KINDS))
        for s in S["labs"]:
            cells = []
            for k in KINDS:
                u = masks[k]
                m, lo, hi = clustered(SIM[split][s][key][u] - SIM[split]["target"][key][u], S["row_tgt"][u])
                cells.append(f"{m:+.4f} [{lo:+.4f}, {hi:+.4f}]")
            print(f"{SHORT[s]:13s}" + "".join(f"{c:>27s}" for c in cells))



=== test, DINOv2: mean similarity to own references
set                 all     no_up     mixed    all_up
real target      0.4215    0.4195    0.4060    0.4335
baseline         0.4349    0.4229    0.4133    0.4590
LoRA@1000        0.4286    0.4205    0.4104    0.4473
LoRA@4000        0.4130    0.4091    0.4017    0.4236
QLoRA@2000       0.4283    0.4190    0.4094    0.4483
SP(iii)@2000     0.4116    0.4062    0.4019    0.4225
rows                488       157       133       198
targets             100        85        55        70
--- test, DINOv2: generated set minus real target, mean [95% CI, target-clustered]
set                                  all                      no_up                      mixed                     all_up
baseline      +0.0134 [+0.0029, +0.0239] +0.0034 [-0.0123, +0.0190] +0.0073 [-0.0044, +0.0190] +0.0254 [+0.0066, +0.0442]
LoRA@1000     +0.0071 [-0.0029, +0.0170] +0.0010 [-0.0132, +0.0152] +0.0044 [-0.0066, +0.0153] +0.0137 [-0.0043, +0.0317]
LoRA@4000   